# 01 · Benchmark: guards, regression panel, seed noise, held-out tmQM

Runs named **solver configurations** × **seeds** × **case sets** through the standard panel runner (`tests/regression/run_panel.py`: one subprocess per case, hard timeout, MACE-POLAR-1 single points, MACE only measures). Everything is written to `RUNS` with its fingerprint (src hash, DB, solve() arguments, seed); finished runs are reused, so re-running the notebook only computes what is missing.

**Case sets**
- `panel`: 8 DOE aqua (DFT refs), 13 tmQM (VUCVUN + 12 random), 3 legacy guards (Co-EDTA, La-EDTA(H₂O)₃ CN9, Co(en)₃).
- `heldout28`: the 40-complex random tmQM sample of 2026-09-23 minus the 12 panel complexes. Never used for diagnosis or tuning, so it is the honest test of generalisation.

**Why seeds:** with ~13 cases a single run cannot separate a real ±30 kJ/mol change from which conformer happened to win (GUJCUJ moved 66 → 172 kJ/mol under a change that shifted its Q1 targets by ≤ 0.06 Å). Compare configurations per case against the seed spread (`bench.paired_diff`).

**Runtime** (laptop, 2 parallel cases): panel ≈ 5–10 min per configuration per seed; held-out 28 ≈ 10 min. Figures: `02_visualization.ipynb`.

In [ ]:
import sys, os, json, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'jacobik').exists())
sys.path.insert(0, str(ROOT / 'src')); sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from tools.jdev import bench, env
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 200)

DB   = Path(os.environ.get('JACOBIK_DB', ROOT / 'data' / 'refit' / 'final' / 'constraints_refit.json'))
MACE = os.environ.get('JACOBIK_MACE_MODEL_PATH', str(ROOT / 'script_notebooks' / 'MACE-POLAR-1-M.model'))
os.environ['JACOBIK_MACE_MODEL_PATH'] = MACE
RUNS = ROOT / 'data' / 'benchmark_runs' / time.strftime('%Y-%m-%d')   # change to reuse an earlier day's runs
RUNS.mkdir(parents=True, exist_ok=True)
SEEDS = [11, 12, 13]
JOBS = 2                     # cases in parallel
RUN_GUARDS, RUN_PANEL, RUN_HELDOUT = True, True, True
PANEL_CASES = None           # e.g. ['VUCVUN', 'DEFCPO'] for a quick look (stored under a separate tag)
HELD_CASES = None            # None = all 28
HELD_SEEDS = SEEDS           # held-out 28 x configs x seeds: ~10 min per (config, seed) on a laptop

# Named solver configurations = extra solve() keyword arguments (defaults: chelate_embed=True,
# orientation_model='auto' -> S_orient when the DB has QO fits, angle_bends=False).
CONFIGS = {
    'v2':       {},
    'v2+bends': {'angle_bends': True},
    'v1-like':  {'chelate_embed': False, 'orientation_model': 'q3q4'},   # old embed + Q3/Q4 (the Q1 trans-key fix cannot be switched off)
}
print(json.dumps(env.fingerprint(DB, Path(MACE)), indent=1))
from jacobik.stats.stats import load_db
print('QO (S_orient) fits in DB:', len(load_db(DB).data.get('QO', {})), '-> v2 uses', 'S_orient' if load_db(DB).data.get('QO') else 'Q3/Q4 (refit with 03_full_refit to get S_orient)')

## 1. Guards (≈ 2 min): must be green before any numbers count

In [ ]:
if RUN_GUARDS:
    r = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '--ignore=tests/regression'],
                       env=dict(os.environ, JACOBIK_TEST_DB=str(DB)), capture_output=True, text=True)
    print(r.stdout[-1500:]); assert r.returncode == 0, 'guards failed -- fix before benchmarking' 

## 2. Regression panel: every configuration × seed

In [ ]:
PANEL_PATHS = bench.run_matrix(CONFIGS, SEEDS, DB, RUNS, case_set='panel', cases=PANEL_CASES, jobs=JOBS, save_xyz=True) if RUN_PANEL else sorted(RUNS.glob('panel*__*.json'))
panel = bench.load(PANEL_PATHS)
bench.summary(panel)

In [ ]:
# per case: mean / sd / min / max of |dE| over seeds (kJ/mol)
bench.per_case(panel)

In [ ]:
# is a difference bigger than the seed noise? (clear = |mean paired difference| > 2 x pooled seed sd)
bench.paired_diff(panel, 'v1-like', 'v2')

In [ ]:
bench.paired_diff(panel, 'v2', 'v2+bends')

In [ ]:
# checks that must hold for every configuration: donors all bound, legacy cases solved
chk = panel.groupby(['config', 'case']).agg(status=('status', lambda s: ','.join(sorted(set(s)))),
                                          donors=('n_donors', 'first'), donors_xtal=('n_donors_crystal', 'first'),
                                          max_md_err=('max_md_err', 'max')).reset_index()
chk[(chk.status != 'ok') | (chk.max_md_err > 0.5)]

## 3. Held-out tmQM (28 complexes never looked at)

In [ ]:
HELD_PATHS = bench.run_matrix(CONFIGS, HELD_SEEDS, DB, RUNS, case_set='heldout28', cases=HELD_CASES, jobs=JOBS, save_xyz=True) if RUN_HELDOUT else sorted(RUNS.glob('heldout28*__*.json'))
held = bench.load(HELD_PATHS)
bench.summary(held)

In [ ]:
bench.per_case(held)

## 4. Save the tidy table (read by 02_visualization)

In [ ]:
ALL = pd.concat([panel, held], ignore_index=True)
ALL.to_csv(RUNS / 'all_results.csv', index=False)
print(RUNS / 'all_results.csv', len(ALL), 'rows;', ALL.src_hash.unique(), ALL.db.unique())